In [ ]:
#imports
import pydicom as dcm
import os
import torch
import numpy as np
import pandas as pd
import cv2
from dataclasses import dataclass, field
from copy import deepcopy

dicom_dir_path = './dataset/PINN1'
phantom_tensors_dir = './dataset/phantom_tensors'
patches_np_binary_dir = './dataset/patches_np_binary'
patches_plot_dir = './patches_plots'

os.makedirs(phantom_tensors_dir, exist_ok=True)
os.makedirs(patches_np_binary_dir, exist_ok=True)
os.makedirs(patches_plot_dir, exist_ok=True)

INSERT_RADIUS = 15


/Users/giannis/resnet18/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Data classes
@dataclass
class SliceInfo:
  image: np.ndarray = None
  inserts_patches: list = field(default_factory=list)
  z_mm: float = 0
  thickness_mm: float = 0
  kernel: str = ""
  cx: int = 0
  cy: int = 0
  radius: float = 0
  slope: float = 1
  intercept: float = 0
  pixel_size: float = 0
  patch_size: int = 0

@dataclass
class Metrics:
  contrast: list = field(default_factory=list)
  noise: list = field(default_factory=list)

@dataclass
class ScanSlices:
  inserts_slices: list[SliceInfo] = field(default_factory=list)
  uid: int = 0
  acqui_num: int = 0
  metrics: Metrics = field(default_factory=Metrics)

In [ ]:
#Patch processing
def extract_main_circle(dicom_image: np.ndarray, phantom_threshhold: int = 50) -> tuple[int,int,float,np.ndarray]:
  image_norm = cv2.normalize(dicom_image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

  _, mask = cv2.threshold(image_norm, phantom_threshhold, 255, cv2.THRESH_BINARY)
  contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
  outter_circle = max(contours, key=cv2.contourArea)

  (cx, cy), phantom_radius = cv2.minEnclosingCircle(outter_circle)
  cx = int(cx)
  cy = int(cy)

  return cx,cy,phantom_radius

def find_insterts(
      cx: int,
      cy: int,
      radius: float,
      radius_ratio: float,
      insert_angles: list
) -> list[tuple[int]]:
  insert_radius_big = radius_ratio * radius
  insert_centers = []
  #for each angle in the insert find calculate offset from center
  for angle_deg in insert_angles:
    theta = np.deg2rad(angle_deg)
    x = int(cx + insert_radius_big * np.cos(theta))
    y = int(cy + insert_radius_big * np.sin(theta))
    insert_centers.append((x, y))

  return insert_centers

def extract_patches(insert_centers: list[tuple[int]], image: np.ndarray, patch_size: int = 40) -> list[np.ndarray]:
  patches = []
  half = patch_size // 2
  for i, (x, y) in enumerate(insert_centers):
    x1 = max(0, x - half)
    x2 = min(image.shape[1], x + half)
    y1 = max(0, y - half)
    y2 = min(image.shape[0], y + half)
    patch = {'image': image[y1:y2, x1:x2], 'cx': x - x1, 'cy': y - y1}
    patches.append(patch)

  return patches

In [6]:
def extract_patches_varied(insert_centers, image, patch_size=80):
  patches = []
  radius = INSERT_RADIUS

  positions = [
    lambda x, y: (x - radius, y - radius),  # top-right
    lambda x, y: (x - patch_size + radius, y - radius),  # top-left
    lambda x, y: (x - radius, y - patch_size + radius),  # bottom-right
    lambda x, y: (x - patch_size + radius, y - patch_size + radius),  # bottom-left
  ]

  for pos_fn in positions:
    for (x, y) in insert_centers:
      x1, y1 = pos_fn(x, y)
      patch = {'image': image[y1:y1 + patch_size, x1:x1 + patch_size], 'cx': x - x1, 'cy': y - y1}
      patches.append(patch)

  return patches

In [ ]:
def extract_patches_top_right(insert_centers, image, patch_size=80):
  return extract_patches_at(insert_centers, image, patch_size,
         lambda x, y: (x - INSERT_RADIUS, y - INSERT_RADIUS))

def extract_patches_top_left(insert_centers, image, patch_size=80):
  return extract_patches_at(insert_centers, image, patch_size,
         lambda x, y: (x - patch_size + INSERT_RADIUS, y - INSERT_RADIUS))

def extract_patches_bottom_right(insert_centers, image, patch_size=80):
  return extract_patches_at(insert_centers, image, patch_size,
         lambda x, y: (x - INSERT_RADIUS, y - patch_size + INSERT_RADIUS))

def extract_patches_bottom_left(insert_centers, image, patch_size=80):
  return extract_patches_at(insert_centers, image, patch_size,
         lambda x, y: (x - patch_size + INSERT_RADIUS, y - patch_size + INSERT_RADIUS))

def extract_patches_at(insert_centers, image, patch_size, pos_fn):
  patches = []
  for (x, y) in insert_centers:
    x1, y1 = pos_fn(x, y)
    patch = {'image': image[y1:y1 + patch_size, x1:x1 + patch_size], 'cx': x - x1, 'cy': y - y1}
    patches.append(patch)
  return patches

In [ ]:
#Contrast-Noise Ratio
def cnr(patch_hu: np.ndarray, cx: float,cy: float):
  height, width = patch_hu.shape

  radius = INSERT_RADIUS

  #create grid mask for circular insert
  Y, X = np.ogrid[:height, :width]
  circle_mask = (X - cx)**2 + (Y - cy)**2 <= radius**2

  #separate target from background
  target = patch_hu[circle_mask]
  background = patch_hu[~circle_mask]

  contrast = abs(np.mean(target) - np.mean(background))
  noise = np.sqrt(np.var(target) / target.size + np.var(background) / background.size)

  return contrast, noise

In [ ]:
#functions for organizing slices into scans 
def load_group_dcm_slices(directory: str) -> dict:
  insert_z_max_1mm = -1457.87
  insert_z_min_1mm = -1479.47
  insert_z_max_3mm = -1459.07
  insert_z_min_3mm = -1479.07
  max_slices = 11

  scans = {}

  #Traverse trough all dcm files
  for file in os.listdir(directory):
    phantom_path = os.path.join(directory, file)
    ct_slice = dcm.dcmread(phantom_path)

    if not hasattr(ct_slice, "ImagePositionPatient") or ct_slice.SliceThickness == None:
      continue

    uid = ct_slice.SeriesInstanceUID
    z = float(ct_slice.ImagePositionPatient[2])

    if uid not in scans:
      scans[uid] = ScanSlices()
      scans[uid].uid = uid
      scans[uid].acqui_num = ct_slice.AcquisitionNumber

    slice_info = SliceInfo()
    slice_info.image = ct_slice.pixel_array.astype(np.float32)
    slice_info.z_mm = z
    slice_info.thickness_mm = float(ct_slice.SliceThickness)
    slice_info.kernel = str(ct_slice.ConvolutionKernel)
    slice_info.slope = ct_slice.RescaleSlope
    slice_info.intercept = ct_slice.RescaleIntercept
    slice_info.pixel_size = ct_slice.PixelSpacing[0]
      
    slice_info.cx, slice_info.cy, slice_info.radius = extract_main_circle(slice_info.image)

    if len(scans[uid].inserts_slices) < max_slices and ct_slice.SliceThickness == 1 and (insert_z_min_1mm <= z <= insert_z_max_1mm):
      scans[uid].inserts_slices.append(slice_info)

    if len(scans[uid].inserts_slices) < max_slices and ct_slice.SliceThickness == 3 and (insert_z_min_3mm <= z <= insert_z_max_3mm):
      scans[uid].inserts_slices.append(slice_info)


  return list(scans.values())

def dicom_to_patches(all_scans: list[ScanSlices], patch_size: int, patch_fn: callable) -> None:
  radius_ratio = 0.58
  insert_angles = [1, 60.5, 90.5, 120, 180, 241, 271, 301]

  new_scans = []
  for scan in all_scans:
    new_scan = deepcopy(scan)
    for ct_slice in new_scan.inserts_slices:
      insert_centers = find_insterts(
        ct_slice.cx, 
        ct_slice.cy, 
        ct_slice.radius, 
        radius_ratio, 
        insert_angles
      )

      patches = patch_fn(insert_centers, ct_slice.image, patch_size)
      ct_slice.patch_size = patch_size

      ct_slice.inserts_patches = patches
    new_scans.append(new_scan)
  return new_scans


def calculate_metrics(all_scans: list[ScanSlices]):
  for scan in all_scans:
    #CNR
    for inserts_slice in scan.inserts_slices:
      #Caluclate the values for the patches in HU 
      contrasts, noises = [], []
      for patch in inserts_slice.inserts_patches:
        contrast, noise = cnr(
          patch['image'] * inserts_slice.slope + inserts_slice.intercept, 
          patch['cx'], 
          patch['cy']
        )
        contrasts.append(contrast)
        noises.append(noise)

      scan.metrics.contrast.append(contrasts)
      scan.metrics.noise.append(noises)


def create_tensors(all_scan_patches: list, filename: str, iq_csv_path: str) -> None:
  label_tensors = []
  insert_tensors = []
  settings_df = pd.read_csv(iq_csv_path)
  iq_tensors = []
  st_tensors = []
  acqui_num_tensor = []

  sorted_scans= sorted(all_scan_patches, key=lambda sp: sp.acqui_num)

  for scan in sorted_scans:
    #CNR patches to tensor 
    slice_tensors = []
    for inserts_slice in scan.inserts_slices:
      slice_tensors.append(
        torch.stack([
            torch.from_numpy(p['image']).float() for p in inserts_slice.inserts_patches
        ]).unsqueeze(1)
      )
    insert_tensors.append(torch.stack(slice_tensors))

    # metrics labels 
    contrast_per_insert = np.mean(scan.metrics.contrast, axis=0)
    noise_per_insert = np.mean(scan.metrics.noise, axis=0) 
    
    metrics_per_insert = torch.stack([
      torch.tensor(contrast_per_insert, dtype=torch.float32),
      torch.tensor(noise_per_insert, dtype=torch.float32),
    ], dim=1)

    label_tensors.append(metrics_per_insert) 

    csv_condition = ((settings_df['acqui_num'] == scan.acqui_num)
                     & (settings_df['slice_thickness'] == scan.inserts_slices[0].thickness_mm))
    # IQ level from CSV matched by acqui_num
    iq_row = settings_df[csv_condition]['iq_level'].values[0]
    iq_tensors.append(torch.tensor([iq_row], dtype=torch.float32))

    #Slice thickness from CSV
    slice_thick_row = settings_df[csv_condition]['slice_thickness'].values[0]
    st_tensors.append(torch.tensor([slice_thick_row], dtype=torch.float32))

    acqui_num_tensor.append(scan.acqui_num)


  torch.save(
    {
      'insert_patches': insert_tensors, 
      'labels': torch.stack(label_tensors),
      'iq_level': torch.stack(iq_tensors),
      'slice_thickness': torch.stack(st_tensors),
      'acqui_num': torch.tensor(acqui_num_tensor)
    },
    os.path.join(phantom_tensors_dir,filename + '.pt')
  )

In [ ]:
#Create and save CT scans
experiment_datasets = {
  'exp1_2': ([extract_patches], [40]),
  'exp3': ([
    extract_patches, 
    extract_patches_top_right, 
    extract_patches_top_left, 
    extract_patches_bottom_right,
    extract_patches_bottom_left
  ], [60, 80, 100])
}

all_scans = load_group_dcm_slices(dicom_dir_path)

for exp, (patch_funcs, patch_sizes) in experiment_datasets.items():
  for patch_size in patch_sizes:
    new_scans = []
    for patch_fn in patch_funcs:
      out_scans = dicom_to_patches(all_scans, patch_size, patch_fn)
      new_scans.extend(out_scans)

    calculate_metrics(new_scans)
    create_tensors(
      new_scans, 
      f'cataphan_rois_{exp}_{patch_size}',
      './dataset/iq_level_per_scan.csv'
    )